# ShoeCo: multiperiod production planning

ShoeCo must plan production for the next four months. It begins with 500 pairs of shoes in inventory and 100 workers. Forecast demand is:

| Month | 1 | 2 | 3 | 4 |
| --- | --- | --- | --- | --- |
| Demand (pairs) | 3000 | 5000 | 2000 | 1000 |

Each worker is paid 1500 dollars per month and provides up to 160 regular working hours. Each worker can also work up to 20 overtime hours per month, paid at 13 dollars per hour. Hiring a worker costs 1600 dollars; firing a worker costs 2000 dollars. Each pair of shoes requires four labor hours and 15 dollars of raw materials. Holding inventory costs three dollars per pair remaining at the end of each month.

Demand must be met in its own month; late deliveries are not allowed. Choose production, staffing, overtime, and inventory to minimize total cost.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, DataFrames, and Printf are already included. No external data files are needed.

## Problem data and timing

Hiring and firing occur at the start of a month, so the resulting workforce earns that month's salary and supplies that month's labor. Production is available to fill orders during that same month. Inventory is measured after accounting for demand at month end.

All decisions are continuous in this linear program, including the numbers of workers hired, employed, and fired. We will discuss integer decisions later in the course. There is no required final workforce, no severance charge after the planning horizon, and no value assigned to leftover shoes after month 4. Holding costs still apply to final inventory.

Named parameters collect the numerical inputs in one place. `d` is a one-dimensional demand vector, and `T` sets the number of months. The overtime rate is **13 dollars/hour**, matching the problem statement.

In [ ]:
using JuMP, HiGHS, DataFrames, Printf
import MathOptInterface as MOI

d = [3000, 5000, 2000, 1000]  # Demand in pairs of shoes.
T = length(d)
months = 1:T

initial_inventory = 500
initial_workforce = 100
regular_hours_per_worker = 160
overtime_hours_per_worker = 20
labor_hours_per_pair = 4

material_cost = 15       # Dollars per pair produced.
wage_cost = 1500         # Dollars per worker per month.
overtime_cost = 13      # Dollars per overtime hour.
hiring_cost = 1600       # Dollars per worker hired.
firing_cost = 2000       # Dollars per worker fired.
holding_cost = 3         # Dollars per pair at each month end.

## Decision variables

For each month $t = 1,\ldots,T$, use:

| Variable | Meaning | Units |
| --- | --- | --- |
| $x_t$ | Shoes produced | Pairs |
| $w_t$ | Workers employed after hiring and firing | Workers |
| $o_t$ | Overtime used | Hours |
| $h_t$ | Workers hired at the start of the month | Workers |
| $f_t$ | Workers fired at the start of the month | Workers |
| $i_t$ | Inventory remaining at month end | Pairs |

All variables are nonnegative. In particular, $i_t \geq 0$ ensures that no demand is carried forward as a backlog.

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)  # Remove this line to see the solver log.

@variable(model, x[1:T] >= 0)
@variable(model, w[1:T] >= 0)
@variable(model, o[1:T] >= 0)
@variable(model, h[1:T] >= 0)
@variable(model, f[1:T] >= 0)
@variable(model, i[1:T] >= 0)

## Minimize total cost

The objective includes materials, regular wages, overtime, hiring, firing, and inventory holding costs:

$$
\min \sum_{t=1}^{T}\left(
15x_t + 1500w_t + 13o_t + 1600h_t + 2000f_t + 3i_t
\right).
$$

Regular wages are paid for every employed worker, even when some regular hours are unused. Overtime is an additional hourly expense. Every term in the objective is measured in dollars. Named JuMP expressions let us report each cost component after solving.

In [ ]:
@expression(model, material_expense, material_cost * sum(x))
@expression(model, wage_expense, wage_cost * sum(w))
@expression(model, overtime_expense, overtime_cost * sum(o))
@expression(model, hiring_expense, hiring_cost * sum(h))
@expression(model, firing_expense, firing_cost * sum(f))
@expression(model, holding_expense, holding_cost * sum(i))

@objective(model, Min,
    material_expense + wage_expense + overtime_expense +
    hiring_expense + firing_expense + holding_expense)

## Labor capacity and balances across months

Production cannot use more labor than the workforce and overtime provide:

$$
4x_t \leq 160w_t + o_t, \qquad o_t \leq 20w_t
\quad \forall t.
$$

The first inequality allows unused regular hours. The second links the overtime limit to the workforce employed that month.

Inventory and workforce carry information from one month to the next. With initial values $i_0 = 500$ and $w_0 = 100$,

$$
i_{t-1} + x_t = d_t + i_t, \qquad
w_{t-1} + h_t - f_t = w_t
\quad \forall t.
$$

We write the first-month balances separately to insert the initial inventory and workforce. The remaining months use the previous month's decision variables.

Nonnegative inventory in every month forces cumulative production plus the initial stock to cover cumulative demand. Building extra shoes in an earlier month can therefore reduce the need for later overtime or hiring.

In [ ]:
@constraint(model, production[t in months],
    labor_hours_per_pair * x[t] <= regular_hours_per_worker * w[t] + o[t])
@constraint(model, overtime[t in months],
    o[t] <= overtime_hours_per_worker * w[t])

@constraint(model, inv_bal_init, initial_inventory + x[1] == d[1] + i[1])
@constraint(model, inv_bal[t in 2:T], i[t - 1] + x[t] == d[t] + i[t])

@constraint(model, work_bal_init, initial_workforce + h[1] - f[1] == w[1])
@constraint(model, work_bal[t in 2:T], w[t - 1] + h[t] - f[t] == w[t])

model

## Solve and inspect the monthly plan

Check that HiGHS found an optimum and that a feasible solution is available before reading values. The table includes hiring and firing as well as production and staffing, so we can follow both balances month by month.

Keep the unrounded values when checking feasibility. The LP permits fractional workers; rounding decisions independently can break the workforce balances or leave too little production capacity. An implementable plan with whole workers calls for integer restrictions and a new solve, which we will study later.

In [ ]:
optimize!(model)
status = termination_status(model)
println("Termination status: ", status)
status == MOI.OPTIMAL || error("HiGHS stopped with status $(status).")
is_solved_and_feasible(model) || error("No feasible optimal solution is available.")

minimum_cost = objective_value(model)
@printf("\nMinimum total cost: \$%.2f\n", minimum_cost)

plan = DataFrame(
    month = collect(months),
    demand = d,
    produced = value.(x),
    workers = value.(w),
    hired = value.(h),
    fired = value.(f),
    overtime_hours = value.(o),
    inventory = value.(i),
)
plan

## Account for the costs

Each row sums a cost over all months. Compare these components with the monthly plan: salaries are paid even when a worker has idle time, whereas overtime is charged only for the hours used. Holding costs are charged on each month's ending physical inventory.

In [ ]:
cost_report = DataFrame(
    component = ["Materials", "Regular wages", "Overtime", "Hiring", "Firing", "Inventory holding"],
    dollars = [
        value(material_expense),
        value(wage_expense),
        value(overtime_expense),
        value(hiring_expense),
        value(firing_expense),
        value(holding_expense),
    ],
)
cost_report

## Interpret the production plan

With the supplied data, the minimum total cost is **692,500 dollars**.

Use the monthly table to explain how inventory connects the four production decisions. In which months does ShoeCo produce ahead of demand? Which months use overtime, and when does the workforce shrink?

Check that initial inventory plus total production equals total demand plus final inventory. For these data, there is no final inventory, so ShoeCo produces 10,500 pairs over the four months.

Why might ShoeCo retain workers in a low-demand month instead of immediately firing them? Compare the monthly salary with the firing cost, and consider how much of the planning horizon remains. The assumption of no required final workforce or post-horizon firing cost affects that decision.

In [09-ShoeCo-backlog.ipynb](09-ShoeCo-backlog.ipynb), demand can be filled late at a penalty. Predict how this flexibility will affect cost, production, and staffing before solving the next model.

To explore another demand pattern or overtime rate, change the data cell and choose **Run All**. Save a personal copy in `student-work/` if you want to keep your edits.